In [1]:
from sage.all import *

# ------------------------------------------------------------
# Fibonacci polynomial over a base field K:
# F_0 = 0, F_1 = 1, F_n = x^k * F_{n-1} + F_{n-2}
# ------------------------------------------------------------
def fibonacci_poly(k, n, R):
    x = R.gen()

    if k < 0:
        raise ValueError("k must be a non-negative integer.")
    if n < 0:
        raise ValueError("n must be a non-negative integer.")

    if n == 0:
        return R.zero()
    elif n == 1:
        return R.one()

    F0 = R.zero()
    F1 = R.one()

    for _ in range(2, n + 1):
        F0, F1 = F1, x**k * F1 + F0

    return F1


# ------------------------------------------------------------
# Relative Galois data over K = Q(zeta_n)
# ------------------------------------------------------------
def print_relative_galois_data(g, K, label="Polynomial"):
    print(f"{label}: {g}")

    d = g.degree()
    if d <= 0:
        print("Constant polynomial: no nontrivial Galois group.")
        return
    if d == 1:
        print("Linear polynomial: trivial Galois group.")
        print("Galois group structure: trivial")
        print("Order: 1")
        return

    try:
        L.<alpha> = K.extension(g.monic())

        print("Relative extension L/K created successfully.")
        print("L =", L)
        print("Base field K =", K)

        # Relative Galois test
        try:
            is_gal = L.is_galois_relative()
            print("Is Galois over K:", is_gal)
        except Exception as e:
            is_gal = None
            print("Could not test is_galois_relative():", e)

        # Relative automorphisms over the base field
        try:
            autos = L.automorphisms()
            print("Relative automorphisms over K:")
            for sigma in autos:
                print("   ", sigma)
            print("Order of relative automorphism group:", len(autos))
        except Exception as e:
            autos = None
            print("Could not compute relative automorphisms:", e)

        # Try galois_group() if Sage provides it cleanly in your install
        try:
            G = L.galois_group()
            print("Relative Galois group object:", G)

            try:
                print("Galois group structure:", G.structure_description())
            except Exception as e:
                print("Could not compute structure_description():", e)

            try:
                print("Order from group object:", G.order())
            except Exception as e:
                print("Could not compute order from group object:", e)

        except Exception as e:
            print("Could not compute galois_group() directly:", e)
            print("Using relative automorphisms as the reliable fallback.")

    except Exception as e:
        print("Could not create the relative extension K(alpha)/K:", e)


# ------------------------------------------------------------
# Main program:
# given n, use the corresponding cyclotomic field Q(zeta_n)
# ------------------------------------------------------------
k = Integer(input("Enter k (non-negative integer): "))
n_max = Integer(input("Enter maximum n: "))

if k < 0 or n_max < 1:
    raise ValueError("k must be non-negative and n_max must be at least 1.")

for n in range(1, n_max + 1):
    print("\n" + "=" * 100)

    # Base field K_n = Q(zeta_n)
    K = CyclotomicField(n, 'zeta')
    zeta = K.gen()

    # Polynomial ring over K
    R = PolynomialRing(K, 'x')
    x = R.gen()

    # Fibonacci polynomial F_n(x) over K
    f = fibonacci_poly(k, n, R)

    print(f"n = {n}")
    print(f"Base field K = Q(zeta_{n})")
    print(f"Generator zeta = {zeta}")
    print(f"F_({k},{n})(x) over Q(zeta_{n}) = {f}")
    print("Degree:", f.degree())

    if f.is_constant():
        print("Constant polynomial: no nontrivial Galois group.")
        continue

    # Factorization over K
    print("\nFactorization over Q(zeta_n):")
    try:
        fac = f.factor()
        print(fac)
    except Exception as e:
        fac = None
        print("Could not factor over Q(zeta_n):", e)

    # Numerical roots for inspection
    try:
        roots = f.roots(QQbar, multiplicities=False)
        print("\nApproximate roots:")
        print([r.n(12) for r in roots])
    except Exception as e:
        print("Could not compute roots:", e)

    # Whole polynomial if irreducible over K
    if f.is_irreducible():
        print("\nIrreducible over Q(zeta_n): True")
        print_relative_galois_data(f, K, label=f"F_({k},{n})(x)")
    else:
        print("\nIrreducible over Q(zeta_n): False")
        print("Polynomial is reducible over Q(zeta_n).")
        print("Galois data for irreducible factors:")

        if fac is not None:
            for g, mult in fac:
                print(f"\nFactor: {g}  (multiplicity {mult})")
                print_relative_galois_data(g, K, label="Factor")

Enter k (non-negative integer):  1

Enter maximum n:  20

n = 1
Base field K = Q(zeta_1)
Generator zeta = 1
F_(1,1)(x) over Q(zeta_1) = 1
Degree: 0
Constant polynomial: no nontrivial Galois group.

n = 2
Base field K = Q(zeta_2)
Generator zeta = -1
F_(1,2)(x) over Q(zeta_2) = x
Degree: 1

Factorization over Q(zeta_n):
x

Approximate roots:


[0.000]

Irreducible over Q(zeta_n): True
F_(1,2)(x): x
Linear polynomial: trivial Galois group.
Galois group structure: trivial
Order: 1

n = 3
Base field K = Q(zeta_3)
Generator zeta = zeta
F_(1,3)(x) over Q(zeta_3) = x^2 + 1
Degree: 2

Factorization over Q(zeta_n):
x^2 + 1

Approximate roots:
[-1.00*I, 1.00*I]

Irreducible over Q(zeta_n): True
F_(1,3)(x): x^2 + 1
Relative extension L/K created successfully.
L = Number Field in alpha with defining polynomial x^2 + 1 over its base field
Base field K = Cyclotomic Field of order 3 and degree 2
Is Galois over K: True


Relative automorphisms over K:
    Relative number field endomorphism of Number Field in alpha with defining polynomial x^2 + 1 over its base field
  Defn: alpha |--> alpha
        zeta |--> zeta
    Relative number field endomorphism of Number Field in alpha with defining polynomial x^2 + 1 over its base field
  Defn: alpha |--> -alpha
        zeta |--> zeta
Order of relative automorphism group: 2
Relative Galois group object: Galois group 4T2 (2[x]2) with order 4 of x^2 + 1


...

Relative number field endomorphism of Number Field in alpha with defining polynomial x^2 + zeta^17 + zeta^2 + 2 over its base field
  Defn: alpha |--> -alpha
        zeta |--> zeta
Order of relative automorphism group: 2
Relative Galois group object: Galois group of (non-Galois) x^2 + zeta^17 + zeta^2 + 2


Galois group structure: C18 x C2
Order from group object: 36

n = 20
Base field K = Q(zeta_20)
Generator zeta = zeta
F_(1,20)(x) over Q(zeta_20) = x^19 + 18*x^17 + 136*x^15 + 560*x^13 + 1365*x^11 + 2002*x^9 + 1716*x^7 + 792*x^5 + 165*x^3 + 10*x
Degree: 19

Factorization over Q(zeta_n):
x * (x - zeta^6 - zeta^4) * (x - zeta^6 + zeta^4 - 2*zeta^2 + 1) * (x + zeta^6 - zeta^4 + 2*zeta^2 - 1) * (x + zeta^6 + zeta^4) * (x - zeta^7 - zeta^3) * (x - zeta^7 + zeta^5 - zeta^3) * (x + zeta^7 - zeta^5 + zeta^3) * (x + zeta^7 + zeta^3) * (x^2 + 2) * (x^2 - zeta^7 + zeta^3 + 2) * (x^2 - zeta^7 + zeta^5 - zeta^3 + 2*zeta + 2) * (x^2 + zeta^7 - zeta^5 + zeta^3 - 2*zeta + 2) * (x^2 + zeta^7 - zeta^3 + 2)

Approximate roots:
[0.000, -1.98*I, -1.90*I, -1.78*I, -1.62*I, -1.41*I, -1.18*I, -0.908*I, -0.618*I, -0.313*I, 0.313*I, 0.618*I, 0.908*I, 1.18*I, 1.41*I, 1.62*I, 1.78*I, 1.90*I, 1.98*I]

Irreducible over Q(zeta_n): False
Polynomial is reducible over Q(zeta_n).
Galois data for irreducible factors:

Fac

Relative automorphisms over K:
    Relative number field endomorphism of Number Field in alpha with defining polynomial x^2 - zeta^7 + zeta^3 + 2 over its base field
  Defn: alpha |--> alpha
        zeta |--> zeta
    Relative number field endomorphism of Number Field in alpha with defining polynomial x^2 - zeta^7 + zeta^3 + 2 over its base field
  Defn: alpha |--> -alpha
        zeta |--> zeta
Order of relative automorphism group: 2
Relative Galois group object: Galois group of (non-Galois) x^2 - zeta^7 + zeta^3 + 2
Galois group structure: C4 x C2 x C2
Order from group object: 16

Factor: x^2 - zeta^7 + zeta^5 - zeta^3 + 2*zeta + 2  (multiplicity 1)
Factor: x^2 - zeta^7 + zeta^5 - zeta^3 + 2*zeta + 2
Relative extension L/K created successfully.
L = Number Field in alpha with defining polynomial x^2 - zeta^7 + zeta^5 - zeta^3 + 2*zeta + 2 over its base field
Base field K = Cyclotomic Field of order 20 and degree 8
Is Galois over K: True
Relative automorphisms over K:
    Relative numbe

Galois group structure: C4 x C2 x C2
Order from group object: 16

Factor: x^2 + zeta^7 - zeta^5 + zeta^3 - 2*zeta + 2  (multiplicity 1)
Factor: x^2 + zeta^7 - zeta^5 + zeta^3 - 2*zeta + 2
Relative extension L/K created successfully.
L = Number Field in alpha with defining polynomial x^2 + zeta^7 - zeta^5 + zeta^3 - 2*zeta + 2 over its base field
Base field K = Cyclotomic Field of order 20 and degree 8
Is Galois over K: True
Relative automorphisms over K:
    Relative number field endomorphism of Number Field in alpha with defining polynomial x^2 + zeta^7 - zeta^5 + zeta^3 - 2*zeta + 2 over its base field
  Defn: alpha |--> alpha
        zeta |--> zeta
    Relative number field endomorphism of Number Field in alpha with defining polynomial x^2 + zeta^7 - zeta^5 + zeta^3 - 2*zeta + 2 over its base field
  Defn: alpha |--> -alpha
        zeta |--> zeta
Order of relative automorphism group: 2
Relative Galois group object: Galois group of (non-Galois) x^2 + zeta^7 - zeta^5 + zeta^3 - 2*zeta

Relative automorphisms over K:
    Relative number field endomorphism of Number Field in alpha with defining polynomial x^2 + zeta^7 - zeta^3 + 2 over its base field
  Defn: alpha |--> alpha
        zeta |--> zeta
    Relative number field endomorphism of Number Field in alpha with defining polynomial x^2 + zeta^7 - zeta^3 + 2 over its base field
  Defn: alpha |--> -alpha
        zeta |--> zeta
Order of relative automorphism group: 2
Relative Galois group object: Galois group of (non-Galois) x^2 + zeta^7 - zeta^3 + 2
Galois group structure: C4 x C2 x C2
Order from group object: 16
